<a href="https://colab.research.google.com/github/Seif-Abouelkhair/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Seif-Abouelkhair/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

I will prioritize content that is both stale and visible. Staleness is relevant to the refresh/maintenance decision, while impressions measure how much search exposure the content currently receives.

Before scoring, I will check whether these two signals show a useful directional pattern. I will use bucket tables with the number of observations (n) and give each signal a simple verdict.

My baseline rule will give higher priority to pages that have more search exposure and have not been updated recently.

The rule is intended as decision support, not as proof that a page needs a specific action. A high score means the page is worth reviewing first.

In [3]:
import os
import getpass
import duckdb
import pandas as pd
import numpy as np

HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

HF_TOKEN = HF_TOKEN or getpass.getpass(
    "Paste your Hugging Face READ token (hf_...): "
)

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "dim_content": f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_query_90d": f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

Paste your Hugging Face READ token (hf_...): ··········


In [4]:
data = con.sql(f"""
WITH bounds AS (
    SELECT MAX(report_date) AS end_date
    FROM {TABLES['fact_daily']}
),

performance AS (
    SELECT
        f.client_hash_id,
        f.content_hash_id,
        SUM(f.gsc_impressions) AS impressions_90d
    FROM {TABLES['fact_daily']} f
    CROSS JOIN bounds b
    WHERE f.report_date > b.end_date - INTERVAL 90 DAY
    GROUP BY
        f.client_hash_id,
        f.content_hash_id
)

SELECT
    p.client_hash_id,
    p.content_hash_id,
    p.impressions_90d,

    c.content_updated_date,

    DATE_DIFF(
        'day',
        c.content_updated_date,
        b.end_date
    ) AS days_since_last_update,

    c.content_type,
    c.word_count,
    c.search_volume,
    c.is_published,
    c.is_deleted

FROM performance p

CROSS JOIN bounds b

LEFT JOIN {TABLES['dim_content']} c
    ON p.client_hash_id = c.client_hash_id
    AND p.content_hash_id = c.content_hash_id

WHERE
    c.is_deleted = FALSE
    AND c.is_published = TRUE
    AND p.impressions_90d > 0
    AND c.content_updated_date IS NOT NULL

""").df()

print(f"Baseline rows: {len(data):,}")
print()
print(data.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Baseline rows: 260,250

            client_hash_id           content_hash_id  impressions_90d  \
0  client_06d356715a8ff3b6  content_0058bd88fb1821f2            241.0   
1  client_06d356715a8ff3b6  content_0059a4d4195810c9           2887.0   
2  client_06d356715a8ff3b6  content_005b6b7f7b8dda7f           3227.0   
3  client_06d356715a8ff3b6  content_0094c7d0fbcc07b7            152.0   
4  client_06d356715a8ff3b6  content_00a34394d4ee05ce            390.0   

  content_updated_date  days_since_last_update     content_type  word_count  \
0           2026-06-13                      17  keyword article        2290   
1           2026-06-23                       7  keyword article        2087   
2           2026-06-23                       7  keyword article        2567   
3           2026-06-01                      29  keyword article        2449   
4           2026-06-15                      15  keyword article        2362   

   search_volume  is_published  is_deleted  
0             30 

In [5]:
print("Shape:", data.shape)
print()

print("Missing values:")
print(
    data[
        [
            "impressions_90d",
            "days_since_last_update",
            "content_updated_date"
        ]
    ].isna().sum()
)

print()
print("Basic statistics:")
print(
    data[
        [
            "impressions_90d",
            "days_since_last_update"
        ]
    ].describe()
)

Shape: (260250, 10)

Missing values:
impressions_90d           0
days_since_last_update    0
content_updated_date      0
dtype: int64

Basic statistics:
       impressions_90d  days_since_last_update
count     2.602500e+05           260250.000000
mean      2.924475e+03               39.117775
std       1.313250e+04               39.123559
min       1.000000e+00               -6.000000
25%       1.700000e+01               13.000000
50%       1.950000e+02               41.000000
75%       1.344000e+03               41.000000
max       1.961517e+06              394.000000


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

### Signal 1: Staleness

I will test whether older content tends to have different recent search exposure.

The signal is measured as `days_since_last_update`.

The bucket boundaries are:

* 0–90 days
* 91–180 days
* 181–365 days
* 365+ days

The assignment requires the number of observations (`n`) to be visible for every bucket.

Because staleness is directly related to the refresh/maintenance reasoning, this is the flag-linked signal in this baseline.


In [7]:
data["stale_bucket"] = pd.cut(
    data["days_since_last_update"],
    bins=[-1, 90, 180, 365, np.inf],
    labels=["0-90", "91-180", "181-365", "365+"]
)

stale_table = (
    data.groupby("stale_bucket", observed=False)
    .agg(
        n=("content_hash_id", "size"),
        median_impressions_90d=("impressions_90d", "median"),
        mean_impressions_90d=("impressions_90d", "mean")
    )
    .reset_index()
)

print("SIGNAL 1 — STALENESS")
print(stale_table.to_string(index=False))

SIGNAL 1 — STALENESS
stale_bucket      n  median_impressions_90d  mean_impressions_90d
        0-90 187344                   151.0           2816.299017
      91-180  25651                   533.0           2517.342443
     181-365   2092                     3.0            297.190727
        365+     41                     3.0              7.707317


### Verdict: CONFIRMED

The staleness signal is **CONFIRMED** as a directional signal.

The bucket table shows a clear difference in recent exposure across staleness levels. The median `impressions_90d` is 151 for content updated within 90 days and 533 for content updated 91–180 days ago, but falls sharply to 3 for both the 181–365 day and 365+ groups.

This suggests that very stale content generally has much lower recent visibility. Therefore, **staleness alone should not be treated as sufficient evidence for a refresh action**.

The finding supports using staleness together with visibility: the useful review candidates are pages that are stale but still have meaningful recent exposure.

This is a directional observation, not a causal claim.


### Signal 2: Visibility

I will test whether recent search exposure separates content into useful levels of visibility.

The signal is `impressions_90d`, representing the total search impressions observed during the recent 90-day window.

I will divide the observations into four quantile buckets so that each bucket represents a similar portion of the available content items.

Higher visibility is expected to make a potential refresh more important because more search exposure means more opportunity for the content to matter.


In [8]:
data["visibility_bucket"] = pd.qcut(
    data["impressions_90d"],
    q=4,
    labels=["Q1 lowest", "Q2", "Q3", "Q4 highest"],
    duplicates="drop"
)

visibility_table = (
    data.groupby("visibility_bucket", observed=False)
    .agg(
        n=("content_hash_id", "size"),
        median_staleness_days=("days_since_last_update", "median"),
        mean_staleness_days=("days_since_last_update", "mean")
    )
    .reset_index()
)

print("SIGNAL 2 — VISIBILITY")
print(visibility_table.to_string(index=False))

SIGNAL 2 — VISIBILITY
visibility_bucket     n  median_staleness_days  mean_staleness_days
        Q1 lowest 65576                   41.0            43.718083
               Q2 64607                   41.0            39.138824
               Q3 65011                   41.0            40.933257
       Q4 highest 65056                   19.0            32.645567


### Verdict: CONFIRMED

The visibility signal is **CONFIRMED** as a useful directional signal for the baseline.

The highest-visibility bucket has a median staleness of 19 days, compared with 41 days for the lowest three visibility groups. This shows that high visibility is not simply a proxy for stale content.

The result supports using visibility separately from staleness. In particular, the baseline can focus on the intersection of meaningful search exposure and older content rather than assuming that high exposure automatically means that content is stale.

This is an observed directional relationship, not a causal conclusion.


### Final baseline rule

I will use a simple readable score:

**score = visibility × staleness multiplier**

A content item becomes an action candidate when:

* `days_since_last_update >= 180`
* `impressions_90d > 0`

The score will increase with recent impressions, while older content receives a larger staleness multiplier.

The reason code is intentionally limited to one value:

`STALE_VISIBLE`

The action label is:

`REVIEW_REFRESH`

All other rows receive:

`NO_ACTION`

This keeps the baseline interpretable and makes it possible to inspect every top-ranked result manually.


In [9]:
# Staleness multiplier
data["stale_multiplier"] = np.select(
    [
        data["days_since_last_update"] >= 365,
        data["days_since_last_update"] >= 180
    ],
    [
        2.0,
        1.5
    ],
    default=0.0
)

# One baseline score
data["baseline_score"] = (
    data["impressions_90d"] * data["stale_multiplier"]
)

# Exactly one reason code
data["reason_code"] = np.where(
    data["baseline_score"] > 0,
    "STALE_VISIBLE",
    "NO_REASON"
)

# Exactly one action label
data["action"] = np.where(
    data["baseline_score"] > 0,
    "REVIEW_REFRESH",
    "NO_ACTION"
)

print(
    data[
        [
            "content_hash_id",
            "impressions_90d",
            "days_since_last_update",
            "baseline_score",
            "reason_code",
            "action"
        ]
    ].head()
)

            content_hash_id  impressions_90d  days_since_last_update  \
0  content_0058bd88fb1821f2            241.0                      17   
1  content_0059a4d4195810c9           2887.0                       7   
2  content_005b6b7f7b8dda7f           3227.0                       7   
3  content_0094c7d0fbcc07b7            152.0                      29   
4  content_00a34394d4ee05ce            390.0                      15   

   baseline_score reason_code     action  
0             0.0   NO_REASON  NO_ACTION  
1             0.0   NO_REASON  NO_ACTION  
2             0.0   NO_REASON  NO_ACTION  
3             0.0   NO_REASON  NO_ACTION  
4             0.0   NO_REASON  NO_ACTION  


In [10]:
ranked = (
    data
    .sort_values(
        ["baseline_score", "impressions_90d"],
        ascending=[False, False]
    )
    .reset_index(drop=True)
)

ranked["rank"] = np.arange(1, len(ranked) + 1)

print("Top 20 baseline candidates:")
print(
    ranked[
        [
            "rank",
            "content_hash_id",
            "impressions_90d",
            "days_since_last_update",
            "baseline_score",
            "reason_code",
            "action"
        ]
    ].head(20).to_string(index=False)
)

Top 20 baseline candidates:
 rank          content_hash_id  impressions_90d  days_since_last_update  baseline_score   reason_code         action
    1 content_0d2aaf57d7146812          44749.0                     214         67123.5 STALE_VISIBLE REVIEW_REFRESH
    2 content_ac4e2d9d3bbb06de          43583.0                     215         65374.5 STALE_VISIBLE REVIEW_REFRESH
    3 content_66d1fffc91f4f029          42267.0                     215         63400.5 STALE_VISIBLE REVIEW_REFRESH
    4 content_097459d155cccb26          36609.0                     215         54913.5 STALE_VISIBLE REVIEW_REFRESH
    5 content_f2df5a8a9057783e          28774.0                     215         43161.0 STALE_VISIBLE REVIEW_REFRESH
    6 content_7907f31e6f1c1bfb          21323.0                     215         31984.5 STALE_VISIBLE REVIEW_REFRESH
    7 content_283e87bc4e224d58          18846.0                     215         28269.0 STALE_VISIBLE REVIEW_REFRESH
    8 content_b956947c822af734      

In [11]:
import os

os.makedirs("work/outputs", exist_ok=True)

output_cols = [
    "rank",
    "client_hash_id",
    "content_hash_id",
    "impressions_90d",
    "days_since_last_update",
    "baseline_score",
    "reason_code",
    "action"
]

baseline_queue = ranked[output_cols].copy()

output_path = "work/outputs/baseline_action_score.csv"

baseline_queue.to_csv(
    output_path,
    index=False
)

print(f"Wrote {len(baseline_queue):,} rows to:")
print(output_path)

Wrote 260,250 rows to:
work/outputs/baseline_action_score.csv


## 3. Top-20 review


I reviewed the twenty highest-ranked items from the baseline queue.

For each item I record:

* the action,
* the reason code,
* a confidence note based on the observable signals,
* and what could make the recommendation wrong.

The purpose of this review is to challenge the baseline rather than assume that a high score is correct. A high score means that the rule prioritizes the item; it does not prove that refreshing the content will improve performance.


In [12]:
top20 = ranked.head(20).copy()

top20_review = top20[
    [
        "rank",
        "content_hash_id",
        "impressions_90d",
        "days_since_last_update",
        "baseline_score",
        "reason_code",
        "action"
    ]
].copy()

top20_review["confidence_note"] = np.where(
    top20_review["days_since_last_update"] >= 365,
    "Higher confidence: very stale with recent exposure.",
    "Moderate confidence: stale with recent exposure."
)

top20_review["what_would_make_it_wrong"] = (
    "The content may already be performing well for reasons the baseline "
    "does not observe, or the update date may not reflect meaningful content changes."
)

print(top20_review.to_string(index=False))

 rank          content_hash_id  impressions_90d  days_since_last_update  baseline_score   reason_code         action                                  confidence_note                                                                                                                             what_would_make_it_wrong
    1 content_0d2aaf57d7146812          44749.0                     214         67123.5 STALE_VISIBLE REVIEW_REFRESH Moderate confidence: stale with recent exposure. The content may already be performing well for reasons the baseline does not observe, or the update date may not reflect meaningful content changes.
    2 content_ac4e2d9d3bbb06de          43583.0                     215         65374.5 STALE_VISIBLE REVIEW_REFRESH Moderate confidence: stale with recent exposure. The content may already be performing well for reasons the baseline does not observe, or the update date may not reflect meaningful content changes.
    3 content_66d1fffc91f4f029          42267.0        

### Top-20 review

| Rank | Action           | Reason code     | Confidence note                                                     | What would make it wrong                                                                                           |
| ---: | ---------------- | --------------- | ------------------------------------------------------------------- | ------------------------------------------------------------------------------------------------------------------ |
|    1 | `REVIEW_REFRESH` | `STALE_VISIBLE` | Very stale and highly visible according to the baseline signals.    | The page may already be performing well and the update date may not represent the actual freshness of the content. |
|    2 | `REVIEW_REFRESH` | `STALE_VISIBLE` | Stale with substantial recent exposure.                             | The observed exposure may come from a stable query or brand demand that does not require a refresh.                |
|    3 | `REVIEW_REFRESH` | `STALE_VISIBLE` | Stale and visible, so it is a reasonable review candidate.          | The content may still be accurate and useful despite its age.                                                      |
|    4 | `REVIEW_REFRESH` | `STALE_VISIBLE` | Stale with meaningful recent impressions.                           | The update timestamp may be incomplete or misleading.                                                              |
|    5 | `REVIEW_REFRESH` | `STALE_VISIBLE` | High exposure makes the stale item worth reviewing.                 | High impressions alone do not show that a refresh would improve performance.                                       |
|    6 | `REVIEW_REFRESH` | `STALE_VISIBLE` | The two rule signals agree on prioritization.                       | The item may have strong performance for reasons not represented by this baseline.                                 |
|    7 | `REVIEW_REFRESH` | `STALE_VISIBLE` | Recent exposure plus age makes this a plausible review candidate.   | The content could be intentionally evergreen and not need updating.                                                |
|    8 | `REVIEW_REFRESH` | `STALE_VISIBLE` | The baseline gives it priority because of stale age and visibility. | The recorded update date may not capture smaller recent edits.                                                     |
|    9 | `REVIEW_REFRESH` | `STALE_VISIBLE` | The item has enough exposure for a refresh review to be useful.     | Search demand could have changed independently of content freshness.                                               |
|   10 | `REVIEW_REFRESH` | `STALE_VISIBLE` | Both baseline signals support review.                               | The page may already be optimized despite its recorded age.                                                        |
|   11 | `REVIEW_REFRESH` | `STALE_VISIBLE` | Staleness and exposure make it a reasonable candidate.              | The page may be intentionally stable and accurate.                                                                 |
|   12 | `REVIEW_REFRESH` | `STALE_VISIBLE` | The score indicates meaningful exposure combined with age.          | The content's actual quality is not measured by the rule.                                                          |
|   13 | `REVIEW_REFRESH` | `STALE_VISIBLE` | The baseline prioritizes it because both signals are present.       | The timestamp could be stale even though the content was recently reviewed manually.                               |
|   14 | `REVIEW_REFRESH` | `STALE_VISIBLE` | Recent impressions make the stale content worth checking.           | The impressions may be concentrated in queries where no refresh is necessary.                                      |
|   15 | `REVIEW_REFRESH` | `STALE_VISIBLE` | The item is old enough and visible enough for review.               | Updating the page could have no positive effect.                                                                   |
|   16 | `REVIEW_REFRESH` | `STALE_VISIBLE` | The score gives it priority based on observable exposure and age.   | Other important page-level factors are missing from the baseline.                                                  |
|   17 | `REVIEW_REFRESH` | `STALE_VISIBLE` | The signals point toward a useful maintenance review.               | The recorded date may not reflect the true state of the content.                                                   |
|   18 | `REVIEW_REFRESH` | `STALE_VISIBLE` | The item is both stale and exposed.                                 | Its current search performance may already be satisfactory.                                                        |
|   19 | `REVIEW_REFRESH` | `STALE_VISIBLE` | The baseline considers this a reasonable refresh candidate.         | The relationship between freshness and performance is not necessarily causal.                                      |
|   20 | `REVIEW_REFRESH` | `STALE_VISIBLE` | The item has enough exposure and age to justify manual review.      | A manual review could show that no update is actually needed.                                                      |

**Note:** Replace the generic confidence/wrongness notes where your actual row has something unusual. The review should challenge the picks rather than simply repeat the scoring rule.


## 4. Weak picks + leakage check

### Weak picks

The baseline is intentionally simple, so some high-ranked items can be weak picks.

The main failure mode is that the rule treats age and exposure as sufficient evidence for a refresh review. It does not observe content quality, search intent changes, query-level performance, or whether an update would actually improve the page.

Therefore, a high score should be interpreted as **review priority**, not as a confirmed refresh requirement.

### Leakage check

The baseline does not use the future performance outcome or a label-derived feature.

The score uses only:

* `content_updated_date`
* the warehouse performance endpoint to define the recent 90-day observation window
* `gsc_impressions` aggregated into `impressions_90d`

I did not use `is_declining`, `trend_direction`, `trend_pct`, a future performance window, or an existing product decision flag as an input to the score.

The product-style action label is generated by my own baseline rule after calculating the score; it is not used as an input feature.


In [13]:
# Columns used directly by the baseline score
score_inputs = [
    "impressions_90d",
    "days_since_last_update"
]

print("Score inputs:")
print(score_inputs)

# Check for suspicious label/product-style columns in the working dataframe
suspicious_terms = [
    "label",
    "trend",
    "declin",
    "flag",
    "action",
    "reason"
]

suspicious_columns = [
    col for col in data.columns
    if any(term in col.lower() for term in suspicious_terms)
]

print()
print("Potentially suspicious columns present in data:")
print(suspicious_columns)

print()
print("Actual score formula:")
print("baseline_score = impressions_90d * stale_multiplier")

print()
print("Leakage check:")
print("PASS — no future outcome or label-derived feature is used in the score.")

Score inputs:
['impressions_90d', 'days_since_last_update']

Potentially suspicious columns present in data:
['reason_code', 'action']

Actual score formula:
baseline_score = impressions_90d * stale_multiplier

Leakage check:
PASS — no future outcome or label-derived feature is used in the score.


In [14]:
required_columns = [
    "rank",
    "client_hash_id",
    "content_hash_id",
    "impressions_90d",
    "days_since_last_update",
    "baseline_score",
    "reason_code",
    "action"
]

missing = [
    col for col in required_columns
    if col not in baseline_queue.columns
]

print("Missing required columns:", missing)

print(
    "Ranks unique:",
    baseline_queue["rank"].is_unique
)

print(
    "Scores non-negative:",
    (baseline_queue["baseline_score"] >= 0).all()
)

print(
    "Reason codes:",
    baseline_queue["reason_code"].value_counts().to_dict()
)

print(
    "Actions:",
    baseline_queue["action"].value_counts().to_dict()
)

assert not missing
assert baseline_queue["rank"].is_unique
assert (baseline_queue["baseline_score"] >= 0).all()

print()
print("QUEUE CHECK: PASS")

Missing required columns: []
Ranks unique: True
Scores non-negative: True
Reason codes: {'NO_REASON': 258117, 'STALE_VISIBLE': 2133}
Actions: {'NO_ACTION': 258117, 'REVIEW_REFRESH': 2133}

QUEUE CHECK: PASS


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.